# 02. Slack thread 정규화와 burst

## 학습 목표

- message 하나가 아니라 thread 전체를 안정적인 source ID로 upsert합니다.
- thread를 question, summary, resolution, system, code reference로 정규화합니다.
- 같은 작성자의 연속 message를 burst로 만들고 낮은 signal을 걸러냅니다.

규칙 기반 toy distiller를 사용합니다. production에서는 LLM 출력 schema validation과 원문 attribution 검사가 필요합니다.

In [ ]:
from dataclasses import dataclass, asdict
import re
import json

@dataclass(frozen=True)
class Message:
    event_id: str
    author: str
    text: str
    reactions: int = 0

@dataclass(frozen=True)
class ThreadArtifact:
    source_id: str
    question: str
    summary: str
    resolution: str
    systems: tuple[str, ...]
    code_refs: tuple[str, ...]
    message_count: int

messages = [
    Message("evt-1", "Maya", "Why does restore stall after manifest load on the large cluster?"),
    Message("evt-2", "Owen", "I reproduced the checkpoint issue with 128 shards on the NFS mount."),
    Message("evt-3", "Maya", "The logs stop before cache warmup. ERR_MANIFEST_TIMEOUT appears repeatedly."),
    Message("evt-4", "Maya", "Setting CKPT_PREFETCH=4 fixes the restore on the NFS mount. The default prefetch is too high for this storage path, and the run now completes consistently across the large cluster. We should update the checkpoint runbook and remove the legacy workaround after the next deployment validation.", reactions=4),
    Message("evt-5", "Sam", "sounds good thanks"),
]

In [ ]:
def distill_thread(source_id: str, thread: list[Message]) -> ThreadArtifact:
    """검색용 구조를 만들되 source_id로 원문 연결을 보존합니다."""
    question = next((m.text for m in thread if "?" in m.text), thread[0].text)
    meaningful = [m.text for m in thread if len(m.text) >= 30]
    summary = " ".join(meaningful[:2])
    resolution = next(
        (m.text for m in reversed(thread) if re.search(r"\b(fix|fixes|resolved|setting|set)\b", m.text, re.I)),
        "미해결",
    )
    all_text = " ".join(m.text for m in thread)
    systems = tuple(sorted({name for name in ("NFS", "checkpoint", "restore") if name.lower() in all_text.lower()}))
    code_refs = tuple(sorted(set(re.findall(r"\b[A-Z][A-Z0-9_]{3,}\b", all_text))))
    return ThreadArtifact(source_id, question, summary, resolution, systems, code_refs, len(thread))

artifact = distill_thread("slack:CKPT-SUPPORT:thread-8f42", messages)
print(json.dumps(asdict(artifact), ensure_ascii=False, indent=2))

assert "CKPT_PREFETCH" in artifact.code_refs
assert artifact.message_count == len(messages)

In [ ]:
def make_bursts(thread: list[Message]) -> list[list[Message]]:
    """같은 author의 연속 message를 하나의 burst로 묶습니다."""
    bursts: list[list[Message]] = []
    for message in thread:
        if bursts and bursts[-1][-1].author == message.author:
            bursts[-1].append(message)
        else:
            bursts.append([message])
    return bursts

idf = {
    "ERR_MANIFEST_TIMEOUT": 5.2,
    "CKPT_PREFETCH=4": 4.8,
}

def burst_signals(burst: list[Message]) -> dict[str, float | bool | int]:
    text = " ".join(m.text for m in burst)
    max_idf = max((value for token, value in idf.items() if token in text), default=0.0)
    return {
        "max_idf": max_idf,
        "characters": len(text),
        "reactions": sum(m.reactions for m in burst),
        "rare": max_idf >= 4.0,
        "long": len(text) >= 200,
        "social": any(m.reactions > 0 for m in burst),
    }

def qualifies(signals: dict[str, float | bool | int], threshold: int = 2) -> bool:
    # 원문의 weighted combination을 단순한 3개 signal 합으로 흉내 냅니다.
    return sum(bool(signals[key]) for key in ("rare", "long", "social")) >= threshold

for index, burst in enumerate(make_bursts(messages), start=1):
    signals = burst_signals(burst)
    print(index, burst[0].author, signals, "EMBED" if qualifies(signals) else "FILTER")

In [ ]:
class IdempotentStore:
    def __init__(self):
        self.rows: dict[str, ThreadArtifact] = {}
        self.processed_events: set[str] = set()

    def ingest(self, source_id: str, thread: list[Message]) -> bool:
        new_events = {m.event_id for m in thread} - self.processed_events
        if not new_events:
            return False
        # 새 reply가 오면 thread 전체를 다시 정규화하고 같은 source_id에 upsert합니다.
        self.rows[source_id] = distill_thread(source_id, thread)
        self.processed_events.update(m.event_id for m in thread)
        return True

store = IdempotentStore()
assert store.ingest(artifact.source_id, messages) is True
assert store.ingest(artifact.source_id, messages) is False

edited = messages + [Message("evt-6", "Owen", "Runbook update merged in PR-1842.")]
assert store.ingest(artifact.source_id, edited) is True
assert store.rows[artifact.source_id].message_count == 6
print("저장 row 수:", len(store.rows), "처리 event 수:", len(store.processed_events))

## 확장 과제

1. message edit와 delete event를 반영하는 tombstone을 추가합니다.
2. LLM이 만든 resolution이 원 thread의 어느 message에 근거하는지 source span을 저장합니다.
3. thread마다 다른 ACL이 burst row에도 동일하게 전파되는지 테스트합니다.
4. 한국어 Slack에서는 문자 수 대신 형태소와 token 통계를 사용해 burst threshold를 다시 보정합니다.